In [1]:
# system tools
import sys
from pathlib import Path

# battle processing
import json
from zipfile import ZipFile
from tools.battle import Battle
from tools.full_pokemon import FullPokemon
import copy

# data science
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score,StratifiedKFold
import statsmodels.api as sm

repo = Path.cwd().resolve()
sys.path.append(str(repo / "damage-calc-python-wrapper"))
sys.path.append(str(repo / "damage-calc-python-wrapper" / "python_calc"))

num_training_zips = 3 # change if you want fewer/more battles at the benefit/cost of less/more time; current max is 3
replay_dir = repo / "data" / "replays"
zip_paths = [replay_dir / f"gen9randombattles_{i}.zip" for i in range(1,num_training_zips+1)]
replay_zips = [ZipFile(zip_path,'r') for zip_path in zip_paths]

from python_calc import (
    Pokemon,
    Move,
    Field,
    Side,
    calc,
    advantage
)

## Example Calculations

In [2]:
palafin = Pokemon(
    name="Palafin-Hero",
    gen=9,
    level=100,
    moves=["Bulk Up", "Overheat","Wave Crash","Drain Punch"],
    evs={"hp" : 248, "atk" : 8, "spd" : 252},
    nature="Careful",
    item="Leftovers")

In [3]:
delphox = Pokemon(
    gen=9,
    name="Delphox",
    evs={"spa" : 252, "spe" : 252, "hp" : 4},
    nature = "Jolly",
    moves = ["fireblast"]
)

In [4]:
gengar = Pokemon(
    gen=9,
    name="Gengar",
    level=100,
    item="Choice Specs",
    nature="Timid",
    evs={"spa" : 252, "spe" : 252, "spd" : 4},
    moves = ["Shadow Ball", "Sludge Wave", "Thunderbolt", "Focus Blast"],
    curHP=261)

In [5]:
tauros_p_a = Pokemon(
    name="taurospaldeaaqua",
    gen=9,
    level=100
)

In [6]:
fire_blast = Move(gen=9,name="fireblast")

In [7]:
calc.calculate(gen = 9, attacker = delphox, defender = tauros_p_a, move = fire_blast,field=Field())

{'gen': 9,
 'attacker': {'name': 'Delphox',
  'ability': 'Blaze',
  'item': None,
  'level': 100,
  'nature': 'Jolly',
  'types': ['Fire', 'Psychic'],
  'stats': {'hp': 292,
   'atk': 174,
   'def': 180,
   'spa': 294,
   'spd': 236,
   'spe': 337},
  'rawStats': {'hp': 292,
   'atk': 174,
   'def': 180,
   'spa': 294,
   'spd': 236,
   'spe': 337},
  'boosts': {'hp': 0, 'atk': 0, 'def': 0, 'spa': 0, 'spd': 0, 'spe': 0},
  'originalCurHP': 292,
  'isDynamaxed': None,
  'teraType': None,
  'status': '',
  'toxicCounter': 0},
 'defender': {'name': 'taurospaldeaaqua',
  'ability': 'Intimidate',
  'item': None,
  'level': 100,
  'nature': 'Serious',
  'types': ['Fighting', 'Water'],
  'stats': {'hp': 291,
   'atk': 256,
   'def': 246,
   'spa': 96,
   'spd': 176,
   'spe': 236},
  'rawStats': {'hp': 291,
   'atk': 256,
   'def': 246,
   'spa': 96,
   'spd': 176,
   'spe': 236},
  'boosts': {'hp': 0, 'atk': 0, 'def': 0, 'spa': 0, 'spd': 0, 'spe': 0},
  'originalCurHP': 291,
  'isDynamaxed':

In [8]:
id = "2631360263"
with open("data/replays/gen9-randombattle/gen9randombattle-" + id + ".json") as battle_json:
    data = json.load(battle_json)

team1 = [
    Pokemon(
        name=data["teams_full"][0][mon_name]["speciesId"],
        gen=9,
        level=data["teams_full"][0][mon_name]["level"],
        ability = data["teams_full"][0][mon_name]["ability"],
        item = data["teams_full"][0][mon_name]["item"],
        gender = data["teams_full"][0][mon_name]["gender"],
        ivs = data["teams_full"][0][mon_name]["ivs"],
        evs = data["teams_full"][0][mon_name]["evs"],
        # teraType = data["teams_full"][0][mon_name]["teraType"],
        moves = data["teams_full"][0][mon_name]["moves"]
    )
    for mon_name in data["teams_full"][0].keys()
]

team2 = [
    Pokemon(
        name=data["teams_full"][1][mon_name]["speciesId"],
        gen=9,
        level=data["teams_full"][1][mon_name]["level"],
        ability = data["teams_full"][1][mon_name]["ability"],
        item = data["teams_full"][1][mon_name]["item"],
        gender = data["teams_full"][1][mon_name]["gender"],
        ivs = data["teams_full"][1][mon_name]["ivs"],
        evs = data["teams_full"][1][mon_name]["evs"],
        # teraType = data["teams_full"][1][mon_name]["teraType"], # calculate will assume that the teraType is on
        moves = data["teams_full"][1][mon_name]["moves"]
    )
    for mon_name in data["teams_full"][1].keys()
]

In [9]:
rows = [[team1[i].name] + [advantage(m1=team1[i],m2=team2[j]) for j in range(6)] for i in range(6)]
df = pd.DataFrame(rows,columns=['team1'] + [team2[j].name for j in range(6)])
df

,team1,wigglytuff,lokix,mandibuzz,snorlax,torterra,mienshao
0,delphox,0.828184,1.000000,0.477186,0.349335,1.000000,1.000000
1,indeedeef,0.393842,0.036765,0.115809,-0.197418,0.479550,0.289982
2,skarmory,-0.339033,0.370745,-0.190759,0.451064,0.829255,0.525798
3,wugtrio,0.487618,0.278302,-0.149923,0.389151,0.005896,0.017394
4,taurospaldeaaqua,0.034941,0.601378,0.806348,1.000000,0.433071,0.454724
5,victreebel,0.301724,0.096552,-0.111963,0.619828,0.511853,0.667026


## Comparing new and old

In [10]:
files = [replay_zip.read(file_name) for replay_zip in replay_zips for file_name in replay_zip.namelist()]
rows = []

for file in files:
    try:
        data = json.loads(file)

        # new advantage calcs
        team1 = [
            Pokemon(
                name=data["teams_full"][0][mon_name]["speciesId"],
                gen=9,
                level=data["teams_full"][0][mon_name]["level"],
                ability = data["teams_full"][0][mon_name]["ability"],
                item = data["teams_full"][0][mon_name]["item"],
                gender = data["teams_full"][0][mon_name]["gender"],
                ivs = data["teams_full"][0][mon_name]["ivs"],
                evs = data["teams_full"][0][mon_name]["evs"],
                # teraType = data["teams_full"][0][mon_name]["teraType"],
                moves = data["teams_full"][0][mon_name]["moves"]
            )
            for mon_name in data["teams_full"][0].keys()
        ]

        team2 = [
            Pokemon(
                name=data["teams_full"][1][mon_name]["speciesId"],
                gen=9,
                level=data["teams_full"][1][mon_name]["level"],
                ability = data["teams_full"][1][mon_name]["ability"],
                item = data["teams_full"][1][mon_name]["item"],
                gender = data["teams_full"][1][mon_name]["gender"],
                ivs = data["teams_full"][1][mon_name]["ivs"],
                evs = data["teams_full"][1][mon_name]["evs"],
                # teraType = data["teams_full"][1][mon_name]["teraType"], # calculate will assume that the teraType is on
                moves = data["teams_full"][1][mon_name]["moves"]
            )
            for mon_name in data["teams_full"][1].keys()
        ]

        new_advs = [advantage(gen=9,m1=team1[i],m2=team2[j]) for i in range(len(team1)) for j in range(len(team2))]



        # old advantage calcs
        battle = Battle(data_json=data,parse=True)
        team1 = [FullPokemon(battle.teams_full[0][mon]) for mon in battle.teams_full[0].keys()]
        team2 = [FullPokemon(battle.teams_full[1][mon]) for mon in battle.teams_full[1].keys()]

        
        if not battle.custom_ruleQ:
            rows.append({
                    "id": battle.id,
                    "p1": battle.players[0],
                    "p2": battle.players[1],
                    "duration": battle.end_time - battle.start_time,
                    "p1_rating" : battle.player_dets[0]["rating"],
                    "elo_diff": battle.player_dets[0]["rating"] - battle.player_dets[1]["rating"],
                    "p1_wins" : battle.players[0] == battle.winner.name,
                    "p1_revealed_team_size" : len(battle.teams[0].keys()),
                    "p2_revealed_team_size" : len(battle.teams[1].keys()),
                    "new_adv" : sum(new_advs),
                    "old_adv" : sum(FullPokemon.advantage(team1[m1],team2[m2]) for m1 in range(6) for m2 in range(6))
                })
    except (json.JSONDecodeError,UnicodeDecodeError):
        continue

full_match_data = pd.DataFrame(rows)

ss = StandardScaler()
copy = copy.deepcopy(full_match_data)
full_match_data['normalized_elo_diff'] = ss.fit_transform(copy[['elo_diff']])
full_match_data['normalized_new_adv'] = ss.fit_transform(copy[['new_adv']])
full_match_data['normalized_old_adv'] = ss.fit_transform(copy[['old_adv']])

In [11]:
# We should throw away matches where people rage quit early
complete_matches = full_match_data[(full_match_data['duration'] > 60) & ((full_match_data["p1_revealed_team_size"] > 2) | (full_match_data["p2_revealed_team_size"] > 2))]
# This is to grab matches where we know that the players understand the basic switching strategy (from Marz' work on switching)
threshold = 1965
highly_rated_matches = complete_matches[(complete_matches['p1_rating'] > threshold) & (complete_matches[['p1_rating','elo_diff']].sum(axis=1) > threshold)]

In [12]:
highly_rated_matches

,id,p1,p2,duration,p1_rating,elo_diff,p1_wins,p1_revealed_team_size,p2_revealed_team_size,new_adv,old_adv,normalized_elo_diff,normalized_new_adv,normalized_old_adv
3,gen9randombattle-2631529004,WhatEver2102,Duck Cop,123,1999,17,True,1,3,8.340932,6.441962,0.519992,1.548752,0.990985
4,gen9randombattle-2631993792,monomythic,OverthereStair,301,2120,58,False,6,6,-0.922874,1.231337,1.234330,-0.181758,0.181047
7,gen9randombattle-2631439736,Mr Brightside,indias last hope,448,2115,-144,False,6,6,7.048926,6.048035,-2.285091,1.307401,0.929753
8,gen9randombattle-2631771408,Illuminating_Fate,medo6037,287,2170,-7,True,5,4,-1.293582,4.106191,0.101843,-0.251008,0.627914
14,gen9randombattle-2631594339,szbsb,Bigoleg,417,2047,-36,True,5,6,1.553845,1.795251,-0.403420,0.280901,0.268702
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12749,gen9randombattle-2642114009,forcemajor14,majex,196,2084,2,False,6,6,-0.687987,-0.835965,0.258649,-0.137880,-0.140294
12756,gen9randombattle-2642143701,forcemajor14,lexam22,246,2235,-12,False,6,1,5.259238,0.514204,0.014729,0.973081,0.069576
12759,gen9randombattle-2641938445,andonibavi,qiuescent,277,2173,-43,True,5,5,-5.738979,-2.654992,-0.525380,-1.081422,-0.423043
12763,gen9randombattle-2642095210,notmetbh102,Uday30,187,2220,-49,False,6,5,4.724409,6.974347,-0.629918,0.873173,1.073739


In [13]:
# Let's check the p-values of each individual feature in a logistic regression
features = ['normalized_new_adv', 'normalized_old_adv', 'normalized_elo_diff']

models = [sm.Logit(highly_rated_matches['p1_wins'],highly_rated_matches[[feature]],offset=0).fit(disp=False) for feature in features]

rows = []
for i in range(len(features)):
    summary = pd.Series({"feature" : features[i], "p-value" : models[i].pvalues.iloc[0], "coefficient" : models[i].params.iloc[0]})
    rows.append(summary)

table = pd.DataFrame(rows)
table.sort_values("p-value",ascending=True)

,feature,p-value,coefficient
1,normalized_old_adv,7.124808e-07,0.143459
0,normalized_new_adv,1.527312e-06,0.137458
2,normalized_elo_diff,9.414981e-04,0.084737


In [14]:
lr = LogisticRegression(C=np.inf,fit_intercept=False,random_state=207,max_iter=1000)
skf = StratifiedKFold(n_splits=5,shuffle=True,random_state=207)
df = highly_rated_matches # good options include: complete_matches, highly_rated_matches

model_info = [
    ('new_adv_only',lr,df[["normalized_new_adv"]]),
    ('old_adv_only',lr,df[["normalized_old_adv"]]),
    ('elo_diff_only',lr,df[["normalized_elo_diff"]]),
    ('new_adv_elo_diff',lr,df[["normalized_new_adv","normalized_elo_diff"]]),
    ('old_adv_elo_diff',lr,df[["normalized_old_adv","normalized_elo_diff"]])
]

for model_name,model,data_set in model_info:
    cv_scores = cross_val_score(estimator=model,X=data_set,y=highly_rated_matches['p1_wins'],cv=skf,n_jobs=-1,scoring="accuracy")
    print(f"The average accuracy score for the {model_name} model is {np.mean(cv_scores)} +- {np.std(cv_scores,ddof=1)}.")
    print()

The average accuracy score for the new_adv_only model is 0.5285978180413864 +- 0.011035071296971968.

The average accuracy score for the old_adv_only model is 0.5321155558416202 +- 0.012153021547571105.

The average accuracy score for the elo_diff_only model is 0.5128439481221639 +- 0.010724067850741165.

The average accuracy score for the new_adv_elo_diff model is 0.5285943852648065 +- 0.003551495222346544.

The average accuracy score for the old_adv_elo_diff model is 0.5358444094015169 +- 0.016143378135797342.

